# Cosim pid: run and inspect
The batch implementation owns setup, checks and cleanup. Edit the arguments
and fixture profile before running; keep comparisons tied to the same model.

## Open a measurement copy
Create this copy with `software/scripts/new_run.py`. Select the Rogue-enabled
kernel. Set `WARM_TDM_PATH` in that kernel's environment when using a checkout.
Start Jupyter in the run directory, or set RUN_DIR to its explicit path.

In [ ]:
import os
import sys
from pathlib import Path

if os.environ.get("WARM_TDM_PATH"):
    checkout = Path(os.environ["WARM_TDM_PATH"]).expanduser().resolve()
    for relative in ["software/python", "firmware/python", "firmware/submodules/surf/python"]:
        sys.path.insert(0, str(checkout / relative))

import warm_tdm_run as runs
RUN_DIR = runs.find_run()  # Or: runs.validate_run("/shared/path/to/run")
print("Measurement directory:", RUN_DIR)

In [ ]:
HOST, PORT = "localhost", 9099
MANIFEST = RUN_DIR / "config" / "simulation.json"
# Copy the manifest for the actual running simulator/server here first.
# See software/cosim/README_cosim.md for required source/build/fixture fields.
connection_args = ["--host", HOST, "--port", str(PORT), "--manifest", str(MANIFEST)]

In [ ]:
result_dir = runs.run_cosim("verify_cosim_pid", RUN_DIR, connection_args + ["--path", "auto", "--behaviors", "steady,step", "--seed-tune-points"])

In [ ]:
import json
reports = [json.loads(path.read_text()) for path in result_dir.rglob("result.json")]
reports

## Explore recorded data
Use the recorded .dat files with operations.plot_stream_data or plot_pid_debug.
Add plots and observations here; leave the original capture files intact.

In [ ]:
list(result_dir.rglob("*.dat"))

In [ ]:
import warm_tdm_api.operations as ops
captures = sorted(result_dir.rglob("*.dat"))
if captures:
    capture = captures[0]  # Select the intended steady/step/flux capture.
    stream = ops.StreamData(str(capture))
    fields = stream.pid.get(0, {}).get(0, {})
    field = "accumError" if "accumError" in fields else "accumErrorFp"
    ops.plot_pid_debug("c0r0", field=field, pid_data_id=stream)